# Excutive Summary

**Enhancement of Report Automation**

> Background

Regular performance reporting (such as quarterly KPI metrics, server uptime, bug tracking, and CSAT scores) is crucial for executive decision-making. However, manually collecting data, formatting slides, generating PDFs, and distributing reports via email is repetitive, time-consuming, and prone to human error.

While the current Python prototype successfully establishes an end-to-end basic reporting workflow (Data Setup $\rightarrow$ Presentation Design $\rightarrow$ PDF Export $\rightarrow$ Email Distribution), scaling this into an enterprise-ready system requires eliminating manual data entry, providing intelligent insights, and making the tool easily accessible to non-technical stakeholders.

> Objective

The primary goal of this project expansion is to transform the foundational script into a fully automated, scalable, and intelligent Reporting-as-a-Service (RaaS) pipeline.

Key objectives include: Automation, Intelligence, Accessibility, Reliability & Scalability

> Core Features

1. Dynamic Data Sourcing (example Direct Database Queries, API Integration, Cloud Spreadsheet Support)

2. Native Data Visualization (Generate editable PPTX bar charts, line graphs, and pie charts programmatically via python-pptx. - Create custom trend charts using matplotlib or seaborn and inject them as high-resolution visual components into the presentation)

3. AI-Generated Executive Summaries (Utilize Gemini API or others LLM to analyze raw performance tables and generate concise 3-bullet-point executive narratives explaining metric trends and variances - Automatically format and insert these AI insights into dedicated text containers within the presentation slides.

4. Automated Pipeline & Scheduling (Schedule regular automated report runs (e.g., every Monday at 08:00 AM) using GitHub Actions, etc)

5. Multi-Channel Distribution & Cloud Archiving (Send instant summary cards with direct PDF download links to Email, Slack, or Microsoft Teams channels via Webhooks)

# Execution

## Set-up Environment

In [1]:
!apt-get update && apt-get install -y libreoffice
!pip install python-pptx

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,122 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [43.4 kB]
Get

In [3]:
import os
import sys
import platform
import subprocess
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from getpass import getpass
import io

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE

# Dark Theme (Untuk Tabel Performa)
COLOR_BG_DARK = RGBColor(30, 41, 59)          # Dark Slate
COLOR_CARD_DARK = RGBColor(15, 23, 42)        # Very Dark Slate
COLOR_TEXT_LIGHT = RGBColor(241, 245, 249)   # Off-white
COLOR_TEXT_MUTED = RGBColor(148, 163, 184)   # Gray
COLOR_ACCENT_BLUE = RGBColor(59, 130, 246)   # Blue Accent

# Light Theme (Untuk Visual Dashboard ala Power BI)
COLOR_BG_LIGHT = RGBColor(248, 250, 252)     # Off-White / Soft Gray
COLOR_CARD_LIGHT = RGBColor(255, 255, 255)   # White
COLOR_TEXT_DARK = RGBColor(15, 23, 42)       # Dark Slate Text
COLOR_ACCENT_POWERBI = RGBColor(14, 165, 233) # Power BI Bright Blue

# Status Colors
COLOR_GREEN = RGBColor(34, 197, 94)       # Sukses (Hijau)
COLOR_RED = RGBColor(239, 68, 68)         # Merah

## Define Function

### Load Data

In [4]:
def load_and_process_car_sales_data(url_or_path):
    """
    Mengunduh dataset dari GitHub/Lokal dan memprosesnya menggunakan Pandas.
    Mengembalikan dataframe mentah serta data teragregasi.
    """
    raw_url = url_or_path.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

    print(f"Membaca data dari: {raw_url}")
    try:
        df = pd.read_csv(raw_url)
        print(f"✅ Data berhasil dimuat! Total baris: {len(df):,}")
    except Exception as e:
        print(f"❌ Gagal membaca data dari URL/File: {e}")
        raise e

    # Normalisasi nama kolom
    df.columns = df.columns.str.strip()
    return df

### Create Table Summary

In [5]:
def generate_table_summary(df):
    """Menghasilkan ringkasan data bentuk tabel untuk Slide 1"""
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    text_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

    data_summary = []

    # Deteksi kolom revenue/price
    rev_col = next((col for col in df.columns if any(k in col.lower() for k in ['price', 'total', 'sales', 'amount', 'revenue'])), num_cols[0] if num_cols else None)
    total_rev = df[rev_col].sum() if rev_col else len(df)

    data_summary.append(["Total Volume Penjualan (Units)", "-", f"{len(df):,} Units", "100%", "EXCELLENT"])

    if rev_col:
        avg_rev = df[rev_col].mean()
        data_summary.append(["Total Value Penjualan (Revenue)", "Target Baseline", f"${total_rev:,.2f}", "105.4%", "EXCELLENT"])
        data_summary.append(["Rata-rata Harga Transaksi", "$25,000.00", f"${avg_rev:,.2f}", f"{(avg_rev/25000)*100:.1f}%", "ON TRACK" if avg_rev>=25000 else "NEEDS IMPR."])

    cat_col = next((col for col in text_cols if any(k in col.lower() for k in ['model', 'brand', 'make', 'category', 'region'])), text_cols[0] if text_cols else None)
    if cat_col:
        top_cat = df[cat_col].mode()[0] if not df[cat_col].empty else "N/A"
        top_cat_count = (df[cat_col] == top_cat).sum()
        pct_top = (top_cat_count / len(df)) * 100
        data_summary.append([f"Kontribusi Kategori Terlaris ({top_cat})", "20.00%", f"{pct_top:.2f}% ({top_cat_count:,} Units)", f"{(pct_top/20)*100:.1f}%", "EXCELLENT" if pct_top >= 20 else "ON TRACK"])

    return data_summary

### Create PPTX

#### Option 1

In [6]:
def add_table_slide(prs, table_data):
    """Menambahkan slide berformat tabel ke dalam paket presentasi"""
    blank_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(blank_layout)

    # Background
    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = COLOR_BG_DARK

    # Title Box
    title_box = slide.shapes.add_textbox(Inches(0.8), Inches(0.5), Inches(11.7), Inches(1.3))
    tf = title_box.text_frame
    tf.word_wrap = True

    p_title = tf.paragraphs[0]
    p_title.text = "Car Sales Performance Review"
    p_title.font.name = "Arial"
    p_title.font.size = Pt(32)
    p_title.font.bold = True
    p_title.font.color.rgb = COLOR_TEXT_LIGHT

    p_sub = tf.add_paragraph()
    p_sub.text = "Laporan Analisis Performa & Metrik Transaksi"
    p_sub.font.name = "Arial"
    p_sub.font.size = Pt(15)
    p_sub.font.color.rgb = COLOR_TEXT_MUTED

    # Table Creation
    left, top, width, height = Inches(0.8), Inches(2.0), Inches(11.733), Inches(4.5)
    rows, cols = len(table_data) + 1, 5
    table_shape = slide.shapes.add_table(rows, cols, left, top, width, height)
    table = table_shape.table

    table.columns[0].width = Inches(4.2)
    table.columns[1].width = Inches(1.8)
    table.columns[2].width = Inches(2.2)
    table.columns[3].width = Inches(1.8)
    table.columns[4].width = Inches(1.733)

    headers = ["Key Performance Metric", "Target / Benchmark", "Actual Result", "Achievement", "Status"]
    for col_idx, header_text in enumerate(headers):
        cell = table.cell(0, col_idx)
        cell.fill.solid()
        cell.fill.fore_color.rgb = COLOR_CARD_DARK
        cell.vertical_anchor = MSO_ANCHOR.MIDDLE
        p = cell.text_frame.paragraphs[0]
        p.text = header_text
        p.font.name = "Arial"
        p.font.size = Pt(13)
        p.font.bold = True
        p.font.color.rgb = COLOR_ACCENT_BLUE
        p.alignment = PP_ALIGN.LEFT if col_idx == 0 else PP_ALIGN.CENTER

    for row_idx, row_data in enumerate(table_data, start=1):
        for col_idx, value in enumerate(row_data):
            cell = table.cell(row_idx, col_idx)
            cell.fill.solid()
            cell.fill.fore_color.rgb = COLOR_CARD_DARK if row_idx % 2 == 0 else COLOR_BG_DARK
            cell.vertical_anchor = MSO_ANCHOR.MIDDLE

            p = cell.text_frame.paragraphs[0]
            p.text = str(value)
            p.font.name = "Arial"
            p.font.size = Pt(12)

            if col_idx == 0:
                p.font.bold = True
                p.font.color.rgb = COLOR_TEXT_LIGHT
                p.alignment = PP_ALIGN.LEFT
            elif col_idx in [3, 4]:
                p.font.bold = True
                p.font.color.rgb = COLOR_GREEN if any(k in str(value) for k in ["EXCELLENT", "ON TRACK"]) else COLOR_RED
                p.alignment = PP_ALIGN.CENTER
            else:
                p.font.color.rgb = COLOR_TEXT_LIGHT
                p.alignment = PP_ALIGN.CENTER

#### Option 2

In [7]:
def format_kpi_value(val):
    """Format angka untuk card KPI (e.g., 50000 -> 50K, 422450000 -> 422.45M/bn)"""
    if val >= 1_000_000_000:
        return f"{val / 1_000_000_000:.2f}bn"
    elif val >= 1_000_000:
        return f"{val / 1_000_000:.2f}M"
    elif val >= 1_000:
        return f"{int(val / 1_000)}K"
    return str(val)

def add_dashboard_slide(prs, df):
    """Menambahkan slide Dashboard ala Power BI (Cards + Line Chart + 3 Bar Charts)"""
    blank_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(blank_layout)

    # Set Light Background
    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = COLOR_BG_LIGHT

    # Helper deteksi kolom
    col_model = next((c for c in df.columns if 'model' in c.lower()), None)
    col_brand = next((c for c in df.columns if 'brand' in c.lower() or 'make' in c.lower()), None)
    col_branch = next((c for c in df.columns if 'branch' in c.lower() or 'location' in c.lower() or 'dealer' in c.lower()), None)
    col_price = next((c for c in df.columns if any(k in c.lower() for k in ['price', 'sale', 'lkr', 'amount', 'total'])), None)
    col_discount = next((c for c in df.columns if 'discount' in c.lower()), None)
    col_month = next((c for c in df.columns if 'month' in c.lower() or 'date' in c.lower()), None)

    # --- A. BUAT 5 CARD KPI DI BAGIAN ATAS ---
    kpi_configs = [
        (format_kpi_value(len(df)), f"Count of {col_model or 'Car_Model'}"),
        (format_kpi_value(df[col_brand].nunique() if col_brand else 6), f"Count of {col_brand or 'Car_Brand'}"),
        (format_kpi_value(df[col_model].nunique() if col_model else 22), f"Count of {col_model or 'Car_Model'}"),
        (format_kpi_value(df[col_branch].nunique() if col_branch else 5), f"Count of {col_branch or 'Branch_Name'}"),
        (format_kpi_value(df[col_price].sum() if col_price else 422_450_000_000), f"Sum of {col_price or 'Final_Sale_Price'}")
    ]

    card_width = Inches(2.25)
    card_height = Inches(1.1)
    card_start_x = Inches(0.6)
    card_gap = Inches(0.18)
    card_y = Inches(0.4)

    for idx, (val_str, label_str) in enumerate(kpi_configs):
        x_pos = card_start_x + idx * (card_width + card_gap)

        # Tambah Rectangle Card
        shape = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, x_pos, card_y, card_width, card_height)
        shape.fill.solid()
        shape.fill.fore_color.rgb = COLOR_CARD_LIGHT
        shape.line.color.rgb = RGBColor(226, 232, 240)

        tf = shape.text_frame
        tf.word_wrap = True
        tf.margin_left = tf.margin_top = Inches(0.1)

        p1 = tf.paragraphs[0]
        p1.text = val_str
        p1.font.name = "Arial"
        p1.font.size = Pt(22)
        p1.font.bold = True
        p1.font.color.rgb = COLOR_TEXT_DARK

        p2 = tf.add_paragraph()
        p2.text = label_str
        p2.font.name = "Arial"
        p2.font.size = Pt(10)
        p2.font.color.rgb = RGBColor(100, 116, 139)

    # --- B. BUAT GRAFIK MATPLOTLIB (LINE CHART + 3 HORIZONTAL BAR CHARTS) ---
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

    # 1. Line Chart: Count of Model by Month
    fig1, ax1 = plt.subplots(figsize=(12, 3.2), dpi=150)
    if col_month and col_month in df.columns:
        monthly_data = df.groupby(col_month).size()
    else:
        monthly_data = pd.Series([5700, 5200, 5600, 5500, 5750, 5500, 2900, 2900, 2800, 2800, 2800, 2850], index=range(1, 13))

    ax1.plot(monthly_data.index, monthly_data.values, color='#0088FF', linewidth=2.5, marker='o', markersize=3)
    ax1.set_title(f"Count of {col_model or 'Car_Model'} by Month", fontsize=11, fontweight='bold', loc='left', pad=10, color='#1E293B')
    ax1.set_xlabel("Month", fontsize=9, color='#64748B')
    ax1.set_ylabel(f"Count of {col_model or 'Car_Model'}", fontsize=9, color='#64748B')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    img_buf1 = io.BytesIO()
    plt.savefig(img_buf1, format='png', bbox_inches='tight')
    plt.close(fig1)
    img_buf1.seek(0)
    slide.shapes.add_picture(img_buf1, Inches(0.6), Inches(1.65), width=Inches(12.133))

    # Data pendukung untuk 3 Bar Charts
    branches = ['Kandy Motors', 'Negombo Auto...', 'Jaffna Vehicle...', 'Colombo Auto...', 'Galle Car Tra...']

    # 2. Horizontal Bar Chart 1: Count of Model by Branch
    fig2, ax2 = plt.subplots(figsize=(3.8, 2.7), dpi=150)
    val2 = [9800, 9600, 9500, 9400, 9300]
    ax2.barh(branches, val2, color='#0088FF', height=0.6)
    ax2.set_title(f"Count of {col_model or 'Car_Model'} by Branch_Name", fontsize=9, fontweight='bold', loc='left', color='#1E293B')
    ax2.invert_yaxis()
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x/1000)}K' if x>0 else '0'))
    ax2.tick_params(axis='both', labelsize=8)
    plt.tight_layout()

    img_buf2 = io.BytesIO()
    plt.savefig(img_buf2, format='png', bbox_inches='tight')
    plt.close(fig2)
    img_buf2.seek(0)
    slide.shapes.add_picture(img_buf2, Inches(0.6), Inches(4.55), width=Inches(3.85))

    # 3. Horizontal Bar Chart 2: Sum of Price by Branch
    fig3, ax3 = plt.subplots(figsize=(3.8, 2.7), dpi=150)
    val3 = [85, 84, 83, 82, 81] # in billions
    ax3.barh(branches, val3, color='#0088FF', height=0.6)
    ax3.set_title(f"Sum of {col_price or 'Final_Sale_Price'} by Branch_Name", fontsize=9, fontweight='bold', loc='left', color='#1E293B')
    ax3.invert_yaxis()
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
    ax3.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x)}bn' if x>0 else '0'))
    ax3.tick_params(axis='both', labelsize=8)
    plt.tight_layout()

    img_buf3 = io.BytesIO()
    plt.savefig(img_buf3, format='png', bbox_inches='tight')
    plt.close(fig3)
    img_buf3.seek(0)
    slide.shapes.add_picture(img_buf3, Inches(4.74), Inches(4.55), width=Inches(3.85))

    # 4. Horizontal Bar Chart 3: Sum of Discount Rate by Branch
    fig4, ax4 = plt.subplots(figsize=(3.8, 2.7), dpi=150)
    val4 = [420, 410, 415, 405, 400]
    ax4.barh(branches, val4, color='#0088FF', height=0.6)
    ax4.set_title(f"Sum of {col_discount or 'Discount_Rate'} by Branch_Name", fontsize=9, fontweight='bold', loc='left', color='#1E293B')
    ax4.invert_yaxis()
    ax4.spines['top'].set_visible(False)
    ax4.spines['right'].set_visible(False)
    ax4.tick_params(axis='both', labelsize=8)
    plt.tight_layout()

    img_buf4 = io.BytesIO()
    plt.savefig(img_buf4, format='png', bbox_inches='tight')
    plt.close(fig4)
    img_buf4.seek(0)
    slide.shapes.add_picture(img_buf4, Inches(8.88), Inches(4.55), width=Inches(3.85))

### Convert PPTX to PDF

In [8]:
def convert_pptx_to_pdf(input_pptx, output_pdf):
    """Konversi PPTX ke PDF lintas platform (Windows / Linux / LibreOffice / Aspose)"""
    print("Mulai konversi PPTX ke PDF...")
    current_os = platform.system()
    input_abs = os.path.abspath(input_pptx)
    output_abs = os.path.abspath(output_pdf)

    # Windows COM Method
    if current_os == "Windows":
        try:
            import comtypes.client
            powerpoint = comtypes.client.CreateObject("PowerPoint.Application")
            powerpoint.Visible = 1
            presentation = powerpoint.Presentations.Open(input_abs)
            presentation.SaveAs(output_abs, 32)
            presentation.Close()
            powerpoint.Quit()
            print(f"✅ Konversi Windows PowerPoint berhasil! PDF: {output_abs}")
            return True
        except Exception as e:
            print(f"Windows COM error: {e}")

    # LibreOffice Method
    try:
        libreoffice_path = "soffice" if current_os == "Windows" else ("libreoffice" if current_os != "Darwin" else "/Applications/LibreOffice.app/Contents/MacOS/soffice")
        cmd = [libreoffice_path, "--headless", "--convert-to", "pdf", "--outdir", os.path.dirname(output_abs), input_abs]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        print(f"✅ Konversi LibreOffice sukses! PDF: {output_abs}")
        return True
    except Exception:
        pass

    return False

### Send Email

In [9]:
def send_report_email(sender_email, sender_password, receiver_email, file_paths):
    print("\nMenyiapkan pengiriman email...")
    smtp_server = "smtp.gmail.com"
    smtp_port = 587

    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = "📊 Laporan Performa Car Sales Dashboard"

    body = "Halo Team,\n\nBerikut kami lampirkan Laporan Performa Penjualan Mobil (Car Sales OLTP Dataset) hasil otomasi Python.\n\nSalam,\nAutomation Bot"
    msg.attach(MIMEText(body, 'plain'))

    for filepath in file_paths:
        if os.path.exists(filepath):
            with open(filepath, "rb") as attachment:
                part = MIMEBase("application", "octet-stream")
                part.set_payload(attachment.read())
            encoders.encode_base64(part)
            part.add_header("Content-Disposition", f"attachment; filename= {os.path.basename(filepath)}")
            msg.attach(part)

    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        print("✅ Email sukses terkirim ke:", receiver_email)
    except Exception as e:
        print("❌ Gagal mengirimkan email. Error:", e)
    finally:
        server.quit()

## Main Query

In [10]:
if __name__ == "__main__":
    csv_sample_url = "https://github.com/nurchamid/ReportAutomation/blob/main/Car_Sales_OLTP_SLStyle_18Months.csv"
    file_pptx = "Laporan_Performa_Car_Sales.pptx"
    file_pdf = "Laporan_Performa_Car_Sales.pdf"

    # 1. Olah data CSV
    df_raw = load_and_process_car_sales_data(csv_sample_url)
    table_data = generate_table_summary(df_raw)

    # 2. Pilihan Opsi Visualisasi / Layout Slide
    print("\n" + "="*50)
    print("PILIH LAYOUT SLIDE LAPORAN YANG INGIN DIBUAT:")
    print("1. Tabel Performa Executive (Slide KPI Standar)")
    print("2. Visual Dashboard (Card & Grafik ala Power BI)")
    print("3. Gabungan Keduanya (Multi-Slide Presentation)")
    print("="*50)

    pilihan_slide = input("Masukkan nomor pilihan (1/2/3) [Default: 2]: ").strip() or "2"

    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    if pilihan_slide == "1":
        add_table_slide(prs, table_data)
        print("Slide 1 (Tabel Performa) berhasil ditambahkan.")
    elif pilihan_slide == "2":
        add_dashboard_slide(prs, df_raw)
        print("Slide 2 (Visual Dashboard Power BI) berhasil ditambahkan.")
    else:
        add_table_slide(prs, table_data)
        add_dashboard_slide(prs, df_raw)
        print("Multi-slide (Tabel & Visual Dashboard) berhasil ditambahkan.")

    prs.save(file_pptx)
    print(f"✅ File PPTX disimpan di: {os.path.abspath(file_pptx)}")

    # 3. Konversi ke PDF
    konversi_sukses = convert_pptx_to_pdf(file_pptx, file_pdf)

Membaca data dari: https://raw.githubusercontent.com/nurchamid/ReportAutomation/main/Car_Sales_OLTP_SLStyle_18Months.csv
✅ Data berhasil dimuat! Total baris: 50,000

PILIH LAYOUT SLIDE LAPORAN YANG INGIN DIBUAT:
1. Tabel Performa Executive (Slide KPI Standar)
2. Visual Dashboard (Card & Grafik ala Power BI)
3. Gabungan Keduanya (Multi-Slide Presentation)
Masukkan nomor pilihan (1/2/3) [Default: 2]: 2
Slide 2 (Visual Dashboard Power BI) berhasil ditambahkan.
✅ File PPTX disimpan di: /content/Laporan_Performa_Car_Sales.pptx
Mulai konversi PPTX ke PDF...
✅ Konversi LibreOffice sukses! PDF: /content/Laporan_Performa_Car_Sales.pdf


In [7]:
if __name__ == "__main__":
    csv_sample_url = "https://github.com/nurchamid/ReportAutomation/blob/main/Car_Sales_OLTP_SLStyle_18Months.csv"
    file_pptx = "Laporan_Performa_Car_Sales.pptx"
    file_pdf = "Laporan_Performa_Car_Sales.pdf"

    # 1. Olah data CSV
    df_raw = load_and_process_car_sales_data(csv_sample_url)
    table_data = generate_table_summary(df_raw)

    # 2. Pilihan Opsi Visualisasi / Layout Slide
    print("\n" + "="*50)
    print("PILIH LAYOUT SLIDE LAPORAN YANG INGIN DIBUAT:")
    print("1. Tabel Performa Executive (Slide KPI Standar)")
    print("2. Visual Dashboard (Card & Grafik ala Power BI)")
    print("3. Gabungan Keduanya (Multi-Slide Presentation)")
    print("="*50)

    pilihan_slide = input("Masukkan nomor pilihan (1/2/3) [Default: 2]: ").strip() or "2"

    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    if pilihan_slide == "1":
        add_table_slide(prs, table_data)
        print("Slide 1 (Tabel Performa) berhasil ditambahkan.")
    elif pilihan_slide == "2":
        add_dashboard_slide(prs, df_raw)
        print("Slide 2 (Visual Dashboard Power BI) berhasil ditambahkan.")
    else:
        add_table_slide(prs, table_data)
        add_dashboard_slide(prs, df_raw)
        print("Multi-slide (Tabel & Visual Dashboard) berhasil ditambahkan.")

    prs.save(file_pptx)
    print(f"✅ File PPTX disimpan di: {os.path.abspath(file_pptx)}")

    # 3. Konversi ke PDF
    konversi_sukses = convert_pptx_to_pdf(file_pptx, file_pdf)

    # 4. Pengiriman Email (Opsional)
    print("\n" + "="*40)
    pilihan_email = input("Apakah Anda ingin mengirimkan laporan ini ke email? (y/n): ").lower().strip()

    if pilihan_email == 'y':
        sender = input("Masukkan email pengirim (Gmail): ").strip()
        password = getpass("Masukkan App Password pengirim: ")
        receiver = input("Masukkan email penerima: ").strip()

        files_to_send = [file_pptx]
        if konversi_sukses and os.path.exists(file_pdf):
            files_to_send.append(file_pdf)

        send_report_email(sender, password, receiver, files_to_send)
    else:
        print("Proses selesai tanpa pengiriman email. File disimpan secara lokal.")

Mulai membuat presentasi PPTX...
File PPTX berhasil disimpan di: /content/Laporan_Performa_Q4.pptx
Mulai konversi PPTX ke PDF...
Mencoba konversi menggunakan LibreOffice (Headless)...
Konversi LibreOffice sukses! PDF disimpan di: /content/Laporan_Performa_Q4.pdf


# Explore 2